# DenseNet-121 hecha a mano en ChestX-ray14 — versión del Módulo 8 (PyTorch puro)
**Néstor Issel Hernández Rojo** · Proyecto Final, Módulo 8

> **Versión LeakyReLU.** Es el mismo notebook de `M8_DenseNet121.ipynb` con una sola diferencia: las 121 activaciones ReLU de la DenseNet-121 se cambian por LeakyReLU con pendiente $a=0.01$, $\mathrm{LeakyReLU}(z)=\max(az,\,z)$. Así ninguna neurona tiene gradiente exactamente 0 y no puede quedar "muerta". Se escogió $a=0.01$ porque es la única activación que no rompe lo aprendido en ImageNet: la similitud coseno de los rasgos con la red original es 0.993 con LeakyReLU 0.01, contra 0.391 con 0.1, 0.299 con GELU, 0.277 con SiLU y 0.230 con ELU (`figuras_extra/activacion_similitud.py`). Datos, semilla, receta y evaluación son idénticos; las salidas se llaman `DenseNet121-LeakyReLU`.

**Qué es.** El mismo entrenamiento de `01a_DenseNet121.ipynb` (CNN, 14 enfermedades, separación oficial por paciente), escrito como se escriben los modelos en el Módulo 8: constantes arriba, una clase `Dataset` y una `nn.Module` explícitas, las funciones `train` y `evaluate` escritas a mano, un diccionario `history`, líneas base que no miran la imagen y gráficas con su "cómo leer". La receta es la misma de 01a–01d y las salidas tienen los mismos nombres que lee `02_evaluacion`, así que este notebook puede reemplazar a `01a_DenseNet121.ipynb`.

**Dónde corre.**
- En **Kaggle** (T4) entrena con todos los pacientes. La carpeta `kaggle/DenseNet121` trae este notebook y su `kernel-metadata.json` (kernel privado, GPU, internet y los dos datasets).
- En la **laptop** se activa `MODO_PRUEBA`: pocas imágenes y 1 época, solo para revisar que todo corra.

### Cómo leer este notebook
- Cada celda de código empieza con un bloque **PASO** que dice qué hace y por qué.
- Las celdas **📝** explican con fórmulas y con los números reales del proyecto lo que pasa en la celda anterior.
- Al final, los **Takeaways** y la **📝 Conclusión final**.

### 📝 De dónde sale cada pieza: el mapa del Módulo 8

| Pieza de este notebook | De dónde viene en el módulo |
|---|---|
| Clase `Dataset` con `__init__`, `__len__` y `__getitem__`; `DataLoader` por lotes | Clase 4, *Introduction to PyTorch* (`XORDataset`) |
| `train` y `evaluate` a mano: `zero_grad` → `backward` → `step`, `model.train()` / `model.eval()` y `torch.no_grad()` | Clase 4 (`train_model`, `eval_model`) y Clase 5, *CNNs Fundamentals* (`run_epoch`, `fit`) |
| Early stopping con paciencia y guardar el mejor modelo | Clase 4, *Activation Functions* (`train_model` con `patience`) |
| Bloques de la DenseNet (`DenseLayer`, `DenseBlock`, `TransitionLayer`) y aumento de datos solo en entrenamiento | Clase 5, tutorial *Inception, ResNet and DenseNet* |
| Conv → BatchNorm → ReLU y seguir las formas capa por capa | Clase 5, *CNNs Fundamentals* |
| Constantes arriba, `history`, `smooth`, norma del gradiente, líneas base, "cómo leer la gráfica" y *Takeaways* | Clase 8, *RNN__LSTM* y *LSTM_TS* |

In [ ]:
# ===== PASO 1: Librerías =====
# Qué hace: importa lo mismo que los notebooks del módulo (torch, torch.nn,
#   torch.utils.data, torchvision, matplotlib), más timm (de donde salen los pesos de
#   ImageNet), scikit-learn (AUROC, F1, calibración y separación por paciente) y
#   SummaryWriter para guardar las curvas.
# Por qué: deja listo todo lo que se usa después. SummaryWriter escribe las curvas en el
#   mismo formato que Lightning, así el notebook 02 las lee igual que las de 01a-01d.
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # Windows/Anaconda: torch y numpy traen cada uno su libiomp5md.dll
import inspect
import json
import random
import time
from pathlib import Path

import matplotlib.pyplot as plt
%matplotlib inline
import numpy as np
import pandas as pd
from PIL import Image
import torch
import torch.nn as nn
import torch.utils.data as data
from torch.utils.tensorboard import SummaryWriter
from torchvision import transforms
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, f1_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import GroupShuffleSplit

try:
    import timm
except ModuleNotFoundError:
    !pip install --quiet timm
    import timm

print("torch", torch.__version__, "| timm", timm.__version__)

In [ ]:
# ===== PASO 2: Constantes, semilla y dispositivo =====
# Qué hace: como la celda de configuración de la clase 8 (NUM_EPOCHS, BATCH_SIZE,
#   LEARNING_RATE), reúne arriba todo lo que define el experimento: dónde corre, el
#   modelo, la receta, las rutas y la semilla.
# Por qué: con la receta en un solo lugar se ve que es idéntica a la de 01a-01d. La
#   semilla 42 y cuDNN determinista hacen que dos corridas den lo mismo; en Kaggle, 4
#   procesos preparan los lotes mientras la GPU entrena.
# Dónde corre
EN_KAGGLE = os.path.exists("/kaggle/input")
MODO_PRUEBA = not EN_KAGGLE              # laptop: pocas imágenes y 1 época, solo para revisar que todo corra

# El modelo de este notebook
MODELO = "DenseNet121-LeakyReLU"          # misma red y receta que M8_DenseNet121, con LeakyReLU (PASO 7)
TIMM_NOMBRE = "densenet121.tv_in1k"                 # pesos de ImageNet-1k y normalización, de timm

# Datos y salidas
RAICES_DATOS = [Path("/kaggle/input")] if EN_KAGGLE else [Path("../data"), Path("../kaggle_apoyo")]
CARPETA_SALIDA = Path("/kaggle/working/saved_models") if EN_KAGGLE else Path(
    "./saved_models_m8_prueba" if MODO_PRUEBA else "./saved_models_m8")
CLASES = ["Atelectasis", "Cardiomegaly", "Effusion", "Infiltration", "Mass", "Nodule", "Pneumonia",
          "Pneumothorax", "Consolidation", "Edema", "Emphysema", "Fibrosis", "Pleural_Thickening", "Hernia"]
IMG_SIZE = 224
N_PRUEBA = {"train": 1000, "val": 300, "test": 500}   # tamaños en MODO_PRUEBA (los de 01a)

# Entrenamiento: la receta común de 01a-01d (la configuración D del notebook 03)
NUM_EPOCHS = 1 if MODO_PRUEBA else 10
BATCH_SIZE = 32
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0.05
T_MAX = NUM_EPOCHS                       # coseno: el learning rate llega a casi 0 en la última época
PATIENCE = 3                             # early stopping: épocas sin mejorar el AUROC de validación
FRACCION_TRAIN_EVAL = 0.10               # muestra fija del train para medir su AUROC sin aumentos
NUM_WORKERS = 4 if EN_KAGGLE else 0      # Windows + Jupyter: 0 evita errores de multiprocessing
N_BOOTSTRAP = 20 if MODO_PRUEBA else 200 # remuestreos por paciente para el intervalo de confianza
SEED = 42

CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)


def set_seed(seed):
    """Igual que el tutorial 5: misma semilla en random, numpy y torch (CPU y GPU)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


set_seed(SEED)
torch.backends.cudnn.deterministic = True    # reproducible: la misma corrida da lo mismo
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USA_GPU = device.type == "cuda"
print("Kaggle:", EN_KAGGLE, "| MODO_PRUEBA:", MODO_PRUEBA, "| dispositivo:", device,
      torch.cuda.get_device_name(0) if USA_GPU else "")
print("Salidas en:", CARPETA_SALIDA)

### 📝 Por qué esta receta: lo que ya se probó en el proyecto

| Decisión | Evidencia propia |
|---|---|
| AdamW con $\eta = 10^{-4}$ y decaimiento de pesos $0.05$ (configuración D) | En el notebook 03 (10% de los pacientes, 4 épocas), D dio 0.778 de AUROC de validación y la mejor, C ($3\cdot10^{-4}$), 0.783. La misma D, repetida en el notebook 04, dio 0.773: una diferencia de 0.005 está dentro de lo que varía la misma receta entre corridas. |
| Sin la receta de la clase 8 (Adam, $10^{-3}$) | A dio 0.752 y B (AdamW, $10^{-3}$) 0.722: un paso grande destruye lo que trae ImageNet. |
| Sin recorte del gradiente | E (D + recorte en 1.0) dio 0.776. La norma pasó de 1.0 en el 98% de los pasos y aun así el resultado no cambió, porque AdamW divide cada paso entre la escala de sus gradientes recientes. Aquí la norma solo se **mide**. |
| Sin LeakyReLU ni sesgo inicial | G (LeakyReLU 0.01) dio 0.775 y H (sesgo con la prevalencia) 0.778: ninguno mejoró. |
| Aumento de datos actual, sin volteo | PASO 4 y su 📝. |

**Optimizaciones de velocidad que no cambian el resultado:**
- **Precisión mixta:** la mayoría de las cuentas se hacen en 16 bits.
- **AdamW fusionado:** actualiza todos los parámetros en una sola operación de GPU, si la versión de PyTorch lo permite.
- **Carga en paralelo:** 4 procesos preparan los lotes (`num_workers`), con `pin_memory` y `persistent_workers`.
- **No se usa `channels_last`:** los entrenamientos de 01a–01d no lo usaron, y su efecto en las concatenaciones de la DenseNet no está medido. La semilla y cuDNN determinista se mantienen, porque la reproducibilidad pesa más que unos minutos.

In [ ]:
# ===== PASO 3: Etiquetas, listas oficiales y separación por paciente =====
# Qué hace: lee Data_Entry_2017.csv, convierte 'Finding Labels' (por ejemplo
#   'Effusion|Mass') en 14 columnas de 0/1, separa train_val y test con las listas
#   oficiales del NIH y aparta el 10% de los pacientes de train_val para validación
#   (GroupShuffleSplit con semilla 42). Al final comprueba que ningún paciente esté en dos
#   conjuntos.
# Por qué: es un problema multi-etiqueta (una radiografía puede tener varias enfermedades
#   o ninguna). Separar por paciente evita que el modelo reconozca a la persona en vez de
#   la enfermedad; la separación es idéntica a la de 01a-01d.
def buscar_archivo(patron):
    """Busca un archivo (p. ej. 'Data_Entry_2017*.csv') en RAICES_DATOS sin recorrer las carpetas de imágenes."""
    for raiz in RAICES_DATOS:
        for carpeta, subcarpetas, archivos in os.walk(raiz):
            subcarpetas[:] = [s for s in subcarpetas if not s.startswith("images")]
            for a in archivos:
                if Path(a).match(patron):
                    return Path(carpeta) / a
    raise FileNotFoundError(f"No encontré {patron} en {RAICES_DATOS}")


def buscar_carpeta_imagenes():
    """La primera carpeta que contiene las radiografías .png."""
    for raiz in RAICES_DATOS:
        for carpeta, subcarpetas, archivos in os.walk(raiz):
            if any(a.endswith(".png") for a in archivos[:50]):
                return Path(carpeta)
    raise FileNotFoundError(f"No encontré imágenes .png en {RAICES_DATOS}")


df = pd.read_csv(buscar_archivo("Data_Entry_2017*.csv"))
for c in CLASES:
    df[c] = df["Finding Labels"].str.split("|").apply(lambda etiquetas: int(c in etiquetas))   # 1 si la tiene
IMG_DIR = buscar_carpeta_imagenes()
df["ruta"] = df["Image Index"].apply(lambda nombre: str(IMG_DIR / nombre))

nombres_train_val = set(buscar_archivo("*train_val_list*.txt").read_text().split())
nombres_test = set(buscar_archivo("*test_list*.txt").read_text().split())
df_train_val = df[df["Image Index"].isin(nombres_train_val)].reset_index(drop=True)
df_test = df[df["Image Index"].isin(nombres_test)].reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.10, random_state=SEED)       # 10% de los PACIENTES
idx_train, idx_val = next(gss.split(df_train_val, groups=df_train_val["Patient ID"]))
df_train = df_train_val.iloc[idx_train].reset_index(drop=True)
df_val = df_train_val.iloc[idx_val].reset_index(drop=True)

pacientes = {nombre: set(d["Patient ID"]) for nombre, d in [("train", df_train), ("val", df_val), ("test", df_test)]}
assert not pacientes["train"] & pacientes["val"], "Fuga: pacientes en train y val"
assert not pacientes["train"] & pacientes["test"], "Fuga: pacientes en train y test"
assert not pacientes["val"] & pacientes["test"], "Fuga: pacientes en val y test"

if MODO_PRUEBA:   # subconjunto pequeño, solo para revisar que todo corra
    df_train = df_train.sample(N_PRUEBA["train"], random_state=SEED).reset_index(drop=True)
    df_val = df_val.sample(N_PRUEBA["val"], random_state=SEED).reset_index(drop=True)
    df_test = df_test.sample(N_PRUEBA["test"], random_state=SEED).reset_index(drop=True)

conjuntos = {"entrenamiento": df_train, "validación": df_val, "test oficial": df_test}
pd.DataFrame({"radiografías": [len(d) for d in conjuntos.values()],
              "pacientes": [d["Patient ID"].nunique() for d in conjuntos.values()],
              "% sin hallazgos": [100 * (d[CLASES].sum(axis=1) == 0).mean() for d in conjuntos.values()]},
             index=list(conjuntos)).round(1)

### 📝 Los tres conjuntos, con los números del entrenamiento completo

| Conjunto | Radiografías | Pacientes | Sin hallazgos |
|---|---|---|---|
| Entrenamiento | 77,988 | 25,207 | 58.3% |
| Validación | 8,536 | 2,801 | 58.8% |
| Test oficial | 25,596 | 2,797 | 38.7% |

Con lotes de $B = 32$ y `drop_last=True`, una época son $\lfloor 77{,}988 / 32 \rfloor = 2{,}437$ pasos de gradiente, y 10 épocas, $24{,}370$. El test oficial es otra población: más pacientes hospitalizados, con muchos estudios cada uno y menos radiografías sin hallazgos.

In [ ]:
# ===== PASO 4: Aumento de datos: cada transformación por separado =====
# Qué hace: define las tres transformaciones del entrenamiento: recorte aleatorio de 80% a
#   100% del área, rotación de ±7° y brillo y contraste de ±10%. Luego dibuja una misma
#   radiografía real original, con cada transformación por separado (dos sorteos de cada
#   una) y con las tres juntas, como en los ejemplos de la clase 5.
# Por qué: el aumento le muestra a la red, en cada época, versiones un poco distintas de
#   la misma radiografía; así memoriza menos. Validación y test solo cambian el tamaño: se
#   mide al modelo con las imágenes tal como son.
S = IMG_SIZE
recorte = transforms.RandomResizedCrop((S, S), scale=(0.8, 1.0), ratio=(0.9, 1.1))   # 80%-100% del área
rotacion = transforms.RandomRotation(7)                                             # ángulo al azar en [-7°, 7°]
brillo_contraste = transforms.ColorJitter(brightness=0.1, contrast=0.1)             # factores al azar en [0.9, 1.1]

train_transform = transforms.Compose([recorte, rotacion, brillo_contraste, transforms.ToTensor()])   # solo entrenamiento
test_transform = transforms.Compose([transforms.Resize((S, S)), transforms.ToTensor()])             # validación y test

con_cardiomegalia = df_train[df_train["Cardiomegaly"] == 1]
ejemplo = Image.open((con_cardiomegalia if len(con_cardiomegalia) else df_train)["ruta"].iloc[0]).convert("RGB")
las_tres = transforms.Compose([recorte, rotacion, brillo_contraste])
paneles = [("original", transforms.Resize((S, S))), ("recorte 80-100%", recorte), ("recorte 80-100%", recorte),
           ("rotación ±7°", rotacion), ("rotación ±7°", rotacion),
           ("brillo y contraste ±10%", brillo_contraste), ("brillo y contraste ±10%", brillo_contraste),
           ("las tres juntas", las_tres), ("las tres juntas", las_tres), ("las tres juntas", las_tres)]

torch.manual_seed(SEED)
fig, axes = plt.subplots(2, 5, figsize=(15, 6.6))
for ax, (titulo, t) in zip(axes.flat, paneles):
    ax.imshow(t(ejemplo))
    ax.set_title(titulo, fontsize=10)
    ax.axis("off")
plt.suptitle("Aumento de datos del entrenamiento sobre una misma radiografía (cada panel es un sorteo distinto)")
plt.tight_layout()
plt.show()

### 📝 Qué hace cada transformación y por qué esos rangos
**1. Recorte aleatorio.** Se sortean una fracción del área $s \sim U(0.8,\,1.0)$ y una proporción $r \sim U(0.9,\,1.1)$. El recorte mide
$$w = \sqrt{s\,A\,r}, \qquad h = \sqrt{s\,A / r}$$
donde $A = 224^2$ es el área de la imagen, se coloca en un lugar al azar y se reescala a $224\times224$. Imita pequeñas diferencias de encuadre y de distancia al equipo. Es el mismo recorte del tutorial de DenseNet de la clase 5.

**2. Rotación.** Cada píxel en la posición $(x, y)$ se mueve alrededor del centro $c$ de la imagen con la matriz de rotación:
$$\begin{pmatrix} x' \\ y' \end{pmatrix} = \begin{pmatrix} \cos\theta & -\sin\theta \\ \sin\theta & \cos\theta \end{pmatrix}\begin{pmatrix} x - c_x \\ y - c_y \end{pmatrix} + \begin{pmatrix} c_x \\ c_y \end{pmatrix}, \qquad \theta \sim U(-7^\circ,\, 7^\circ)$$
Imita la inclinación del paciente al colocarse frente a la placa. Las esquinas que quedan vacías se llenan con negro.

**3. Brillo y contraste.** Cambian la intensidad $x \in [0,1]$ de cada píxel con una transformación lineal $x' = a\,x + b$:
$$\text{brillo: } x' = \beta\,x, \qquad \text{contraste: } x' = \gamma\,x + (1-\gamma)\,\mu, \qquad \beta, \gamma \sim U(0.9,\, 1.1)$$
donde $\mu$ es la intensidad media de la imagen. Imitan las diferencias de exposición entre equipos, por ejemplo una radiografía portátil (AP) contra una de pie (PA).

**Por qué no se voltea.** El tutorial de la clase 5 voltea las imágenes porque en CIFAR la orientación no importa, y aclara que sería distinto "if we would try to detect digits or letters". Una radiografía es ese caso: el corazón está a la izquierda y la placa trae letras de plomo. La evidencia propia lo respalda (notebook 05, 30% de los pacientes y 4 épocas). Con el aumento actual y las semillas 42, 43 y 44, DenseNet-121 dio 0.8147, 0.8100 y 0.8054 de AUROC de validación: media $\bar x = 0.8100$ y desviación estándar $s = 0.0046$. Con volteo dio 0.8118, dentro de ese ruido. No mejora, así que no se agrega.

**Por qué sí el aumento.** En el mismo experimento, quitarlo bajó a DenseNet-121 a 0.7989, unas 2.4 desviaciones estándar debajo de la media: $(0.7989 - 0.8100)/0.0046 \approx -2.4$. A DeiT-S lo bajó de 0.7990 a 0.7838. Sin aumento, el modelo memoriza: la brecha entre el AUROC de entrenamiento y el de validación en la última época fue de 0.174 en DenseNet-121, contra 0.092 con aumento.

**Solo en entrenamiento.** "The validation set should not use the augmentation" (clase 5): validación y test miden al modelo con las radiografías tal como llegan.

In [ ]:
# ===== PASO 5: Dataset, DataLoaders y pos_weight =====
# Qué hace: define ChestXrayDataset (como el XORDataset de la clase 4: __init__, __len__ y
#   __getitem__), arma cuatro DataLoaders (entrenamiento con aumento, una muestra fija del
#   10% del entrenamiento sin aumento, validación y test) y calcula el peso de los
#   positivos de cada enfermedad.
# Por qué: el DataLoader entrega lotes de 32 y baraja solo el entrenamiento. La muestra
#   fija sin aumento sirve para comparar el AUROC de entrenamiento con el de validación en
#   las mismas condiciones. pos_weight hace que un positivo de una enfermedad rara pese
#   más en la pérdida.
class ChestXrayDataset(data.Dataset):

    def __init__(self, df, transform):
        """
        Inputs:
            df - tabla con la ruta de cada radiografía y sus 14 etiquetas (0/1)
            transform - transformaciones de torchvision que se aplican a cada imagen
        """
        super().__init__()
        self.rutas = df["ruta"].tolist()
        self.etiquetas = torch.tensor(df[CLASES].values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.rutas)

    def __getitem__(self, idx):
        img = Image.open(self.rutas[idx]).convert("RGB")      # gris -> 3 canales, como esperan los pesos de ImageNet
        return self.transform(img), self.etiquetas[idx]


def cargador(df_conjunto, transform, barajar):
    """DataLoader con lotes de BATCH_SIZE; en Kaggle 4 procesos preparan los lotes mientras la GPU trabaja."""
    return data.DataLoader(ChestXrayDataset(df_conjunto, transform), batch_size=BATCH_SIZE, shuffle=barajar,
                           drop_last=barajar, pin_memory=USA_GPU, num_workers=NUM_WORKERS,
                           persistent_workers=NUM_WORKERS > 0)


df_train_eval = df_train.sample(frac=FRACCION_TRAIN_EVAL, random_state=SEED).reset_index(drop=True)
train_loader = cargador(df_train, train_transform, barajar=True)
train_eval_loader = cargador(df_train_eval, test_transform, barajar=False)   # muestra fija, sin aumento
val_loader = cargador(df_val, test_transform, barajar=False)
test_loader = cargador(df_test, test_transform, barajar=False)

# Balanceo de clases: peso de los positivos = raíz(negativos / positivos), calculado solo con el entrenamiento
positivos = df_train[CLASES].sum().clip(lower=1)
POS_WEIGHT = np.sqrt((len(df_train) - positivos) / positivos).round(2).tolist()
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(POS_WEIGHT, device=device))

print(f"entrenamiento: {len(df_train):,} ({len(train_loader):,} pasos por época) | muestra sin aumento: "
      f"{len(df_train_eval):,} | validación: {len(df_val):,} | test: {len(df_test):,}")
pd.Series(POS_WEIGHT, index=CLASES, name="pos_weight").to_frame().T

### 📝 La pérdida: sigmoide y entropía cruzada binaria con `pos_weight`
Cada una de las 14 salidas es una pregunta independiente de sí o no. La red da un *logit* $z_c$ por enfermedad y la sigmoide lo convierte en probabilidad:
$$\sigma(z) = \frac{1}{1 + e^{-z}}$$
Para una radiografía con etiquetas $y_c \in \{0, 1\}$, la pérdida de la enfermedad $c$ es:
$$\ell_c = -\Big[w_c\, y_c \log \sigma(z_c) + (1 - y_c)\log\big(1 - \sigma(z_c)\big)\Big], \qquad w_c = \sqrt{\frac{N_c^-}{N_c^+}}$$
y `BCEWithLogitsLoss` promedia los $14 \times B$ términos del lote. Es la misma sigmoide con entropía cruzada binaria del XOR de la clase 4, repetida 14 veces.

**Con los números del entrenamiento completo:**
- Infiltration (16% de las radiografías): $w = 2.29$.
- Atelectasis: $w = 3.09$.
- Hernia (0.16%): $w = 25.26$. Sin la raíz pesaría $25.26^2 \approx 638$ veces más que un negativo y el entrenamiento se volvería inestable.
- **Ejemplo**, con un logit $z = -2$, es decir $\sigma(-2) = 0.119$:
  - si la radiografía **tiene** Hernia, $\ell = 25.26 \times (-\log 0.119) = 25.26 \times 2.127 = 53.7$;
  - si **no** la tiene, $\ell = -\log(1 - 0.119) = 0.127$.

Así, no detectar una Hernia cuesta 420 veces más que una falsa alarma pequeña.

In [ ]:
# ===== PASO 6: Líneas base que no miran la imagen =====
# Qué hace: calcula, en validación, qué logran dos reglas que no miran la radiografía:
#   decir siempre 'sano' y dar a cada enfermedad la mejor probabilidad constante para esta
#   pérdida, p·w / (p·w + 1 - p), con la prevalencia p del entrenamiento.
# Por qué: como en la sección 'Naive baselines' de LSTM_TS (clase 8): 'a model that cannot
#   beat these has not learned anything useful'. Su accuracy se dibuja junto a la del
#   modelo en cada época, y su pérdida es el piso que el modelo tiene que bajar.
y_val = df_val[CLASES].values
prevalencia = df_train[CLASES].values.mean(axis=0)                  # frecuencia de cada enfermedad en entrenamiento
w = np.array(POS_WEIGHT)
p_constante = prevalencia * w / (prevalencia * w + 1 - prevalencia)  # minimiza la BCE con pos_weight


def bce_ponderada(p, y):
    """La pérdida del entrenamiento para probabilidades p ya calculadas (promedio de N x 14 términos)."""
    p = np.clip(p, 1e-7, 1 - 1e-7)
    return float(np.mean(-(w * y * np.log(p) + (1 - y) * np.log(1 - p))))


ACC_SIEMPRE_SANO = float((y_val == 0).mean())
lineas_base = pd.DataFrame({
    "AUROC val": [0.5, 0.5],
    "Accuracy val (umbral 0.5)": [ACC_SIEMPRE_SANO, float(((p_constante >= 0.5) == y_val).mean())],
    "Accuracy exacta val (14 de 14)": [float((y_val.sum(axis=1) == 0).mean()), float(((p_constante >= 0.5) == y_val).all(axis=1).mean())],
    "Pérdida val": [np.nan, bce_ponderada(np.tile(p_constante, (len(y_val), 1)), y_val)],
}, index=["Siempre 'sano'", "Mejor probabilidad constante"])
lineas_base.round(4)

**Cómo leer la tabla**
- Cualquier regla que no mira la imagen tiene AUROC 0.5: ordena a los enfermos y a los sanos al azar.
- Su accuracy es altísima: con el entrenamiento completo, decir siempre "sano" acierta 95.6% de las respuestas de validación y 92.5% de las del test oficial, y la radiografía completa en 58.8% de los casos de validación. Por eso la accuracy no sirve para elegir modelos en este problema y la métrica principal es el AUROC.
- La pérdida de la mejor probabilidad constante es la que el modelo debe superar en validación. Si no la baja, no aprendió nada de la imagen.

In [ ]:
# ===== PASO 7: La DenseNet-121 hecha a mano =====
# Qué hace: copia las clases del notebook 00 (DenseLayer, DenseBlock, TransitionLayer y
#   DenseNet121), escritas como las del tutorial de DenseNet de la clase 5: cada pieza es
#   una nn.Module con su __init__ y su forward.
# Por qué: es la CNN del proyecto, construida capa por capa. Con los pesos de ImageNet
#   cargados (PASO siguiente) da exactamente la misma salida que la versión de timm.
# Versión LeakyReLU: cada nn.ReLU() pasa a ser ACTIVACION(), una LeakyReLU con pendiente
#   0.01: para z < 0 deja pasar 0.01*z (y gradiente 0.01) en lugar de 0, así que ninguna
#   neurona se queda sin gradiente. Los pesos de ImageNet son los mismos (no tiene pesos).
PENDIENTE = 0.01


def ACTIVACION():
    """La activación de toda la red: LeakyReLU(z) = max(0.01 z, z)."""
    return nn.LeakyReLU(PENDIENTE)


class DenseLayer(nn.Module):

    def __init__(self, c_in, bn_size, growth_rate):
        """
        Inputs:
            c_in - canales que entran (todos los mapas anteriores del bloque, concatenados)
            bn_size - factor del cuello de botella: la conv 1x1 deja bn_size * growth_rate canales
            growth_rate - canales nuevos que agrega esta capa (k = 32)
        """
        super().__init__()
        self.norm1 = nn.BatchNorm2d(c_in)
        self.relu1 = ACTIVACION()
        self.conv1 = nn.Conv2d(c_in, bn_size * growth_rate, kernel_size=1, bias=False)
        self.norm2 = nn.BatchNorm2d(bn_size * growth_rate)
        self.relu2 = ACTIVACION()
        self.conv2 = nn.Conv2d(bn_size * growth_rate, growth_rate, kernel_size=3, padding=1, bias=False)

    def forward(self, x):
        out = self.conv1(self.relu1(self.norm1(x)))         # cuello de botella 1x1: 4k canales
        out = self.conv2(self.relu2(self.norm2(out)))       # conv 3x3: k canales nuevos
        return torch.cat([x, out], dim=1)                   # concatena [entrada, nuevos]: la idea de DenseNet


class DenseBlock(nn.Sequential):

    def __init__(self, c_in, num_layers, bn_size, growth_rate):
        """num_layers capas densas; la capa i recibe c_in + i * growth_rate canales."""
        super().__init__()
        for layer_idx in range(num_layers):
            self.add_module(f"denselayer{layer_idx + 1}",
                            DenseLayer(c_in=c_in + layer_idx * growth_rate, bn_size=bn_size, growth_rate=growth_rate))


class TransitionLayer(nn.Module):

    def __init__(self, c_in, c_out):
        """BN -> ReLU -> conv 1x1 (mitad de canales) -> average pooling 2x2 (mitad de alto y ancho)."""
        super().__init__()
        self.norm = nn.BatchNorm2d(c_in)
        self.relu = ACTIVACION()
        self.conv = nn.Conv2d(c_in, c_out, kernel_size=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        return self.pool(self.conv(self.relu(self.norm(x))))


class DenseNet121(nn.Module):

    def __init__(self, num_classes=1000, num_layers=(6, 12, 24, 16), bn_size=4, growth_rate=32, c_init=64):
        """Tallo 7x7 -> 4 bloques densos [6, 12, 24, 16] con 3 transiciones -> BN -> ReLU -> promedio global -> lineal."""
        super().__init__()
        self.features = nn.Sequential()
        self.features.add_module("conv0", nn.Conv2d(3, c_init, kernel_size=7, stride=2, padding=3, bias=False))
        self.features.add_module("norm0", nn.BatchNorm2d(c_init))
        self.features.add_module("relu0", ACTIVACION())
        self.features.add_module("pool0", nn.MaxPool2d(kernel_size=3, stride=2, padding=1))
        c_hidden = c_init
        for block_idx, L in enumerate(num_layers):
            self.features.add_module(f"denseblock{block_idx + 1}",
                                     DenseBlock(c_in=c_hidden, num_layers=L, bn_size=bn_size, growth_rate=growth_rate))
            c_hidden = c_hidden + L * growth_rate
            if block_idx < len(num_layers) - 1:
                self.features.add_module(f"transition{block_idx + 1}",
                                         TransitionLayer(c_in=c_hidden, c_out=c_hidden // 2))
                c_hidden = c_hidden // 2
        self.features.add_module("norm5", nn.BatchNorm2d(c_hidden))
        self.relu = ACTIVACION()
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Linear(c_hidden, num_classes)
        self.num_features = c_hidden

    def forward(self, x):
        x = self.features(x)
        x = self.pool(self.relu(x)).flatten(1)
        return self.classifier(x)

In [ ]:
# ===== PASO 8: El modelo: pesos de ImageNet, normalización y recorrido de formas =====
# Qué hace: crear_red arma nuestra DenseNet-121 con los pesos de ImageNet de timm y una
#   capa final de 14 salidas, y comprueba que da lo mismo que la de timm.
#   ModeloNormalizado le antepone la normalización con la que se preentrenó, y el
#   recorrido imprime la forma del tensor en cada etapa.
# Por qué: crear_timm toma los pesos del dataset de apoyo si está (en Kaggle) y si no los
#   descarga; son los mismos. ModeloNormalizado guarda la red como 'model' y la
#   normalización como 'mean' y 'std', igual que Lightning: así el notebook 02 carga estos
#   pesos sin cambios.
def crear_timm(nombre, pretrained=True, **kwargs):
    """timm.create_model, pero si <nombre>.safetensors viene en los datos (dataset de apoyo) carga los pesos de ahí."""
    if pretrained:
        try:
            archivo = buscar_archivo(f"{nombre}.safetensors")
            return timm.create_model(nombre, pretrained=True, pretrained_cfg_overlay=dict(file=str(archivo)), **kwargs)
        except FileNotFoundError:
            pass
    return timm.create_model(nombre, pretrained=pretrained, **kwargs)


def crear_red(pretrained=True):
    """Nuestra DenseNet-121 (PASO anterior) con los pesos de ImageNet de timm y una capa final de 14 salidas."""
    red = DenseNet121(num_classes=1000)
    if pretrained:
        red.load_state_dict(crear_timm(TIMM_NOMBRE, pretrained=True).state_dict(), strict=True)   # las 727 piezas
    red.classifier = nn.Linear(red.num_features, len(CLASES))          # capa nueva: 14 enfermedades
    return red


# Verificación: misma entrada -> misma salida que la DenseNet-121 de timm. La de timm usa ReLU,
# así que la verificación arma la red con ReLU y después se regresa a LeakyReLU.
_activacion_proyecto = ACTIVACION
ACTIVACION = nn.ReLU
oficial = crear_timm(TIMM_NOMBRE, pretrained=True).eval()
nuestra = DenseNet121(num_classes=1000).eval()
nuestra.load_state_dict(oficial.state_dict(), strict=True)
x_prueba = torch.randn(2, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    print(f"Diferencia máxima con la DenseNet-121 de timm: {(nuestra(x_prueba) - oficial(x_prueba)).abs().max().item():.2e}")
del oficial, nuestra, x_prueba
ACTIVACION = _activacion_proyecto                   # de aquí en adelante, LeakyReLU
print("Activación del entrenamiento:", ACTIVACION())


class ModeloNormalizado(nn.Module):

    def __init__(self, red, mean, std):
        """
        Inputs:
            red - la red con 14 salidas
            mean, std - normalización con la que se preentrenó (ImageNet; 0.5 en CoAtNet)
        """
        super().__init__()
        self.model = red                                       # "model": el 02 lee las claves "model.*"
        self.register_buffer("mean", torch.tensor(mean).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor(std).view(1, 3, 1, 1))

    def forward(self, x):
        return self.model((x - self.mean) / self.std)          # logits de las 14 enfermedades


def crear_modelo(pretrained=True):
    cfg = timm.models.get_pretrained_cfg(TIMM_NOMBRE).to_dict()
    return ModeloNormalizado(crear_red(pretrained), cfg["mean"], cfg["std"])


def count_parameters(model):
    """La de la clase 8: cuenta solo los parámetros que se entrenan."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


set_seed(SEED)
modelo = crear_modelo(pretrained=True)
red = modelo.model.eval()                                      # eval: el recorrido no toca las estadísticas de BatchNorm
x = torch.zeros(1, 3, IMG_SIZE, IMG_SIZE)

with torch.no_grad():                                   # recorrido de formas: (N, canales, alto, ancho)
    t = x
    for nombre, capa in red.features.named_children():
        t = capa(t)
        if nombre in ("pool0", "norm5") or nombre.startswith(("denseblock", "transition")):
            print(f"{nombre:12s} -> {tuple(t.shape)}")
    print(f"{'salida':12s} -> {tuple(red(x).shape)}")

print(f"Parámetros entrenables: {count_parameters(modelo):,} | normalización: media "
      f"{[round(v, 3) for v in modelo.mean.flatten().tolist()]}, desviación {[round(v, 3) for v in modelo.std.flatten().tolist()]}")

### 📝 Por dentro de DenseNet-121: concatenar en vez de sumar
En una ResNet cada bloque **suma** su entrada: $x_\ell = H_\ell(x_{\ell-1}) + x_{\ell-1}$. En DenseNet cada capa recibe la **concatenación** de todos los mapas anteriores de su bloque:

$$x_\ell = H_\ell\big([x_0, x_1, \dots, x_{\ell-1}]\big), \qquad H_\ell = \text{BN} \to \text{ReLU} \to \text{conv}_{1\times1}(4k) \to \text{BN} \to \text{ReLU} \to \text{conv}_{3\times3}(k)$$

con tasa de crecimiento $k = 32$. La capa $\ell$ de un bloque recibe $k_0 + (\ell - 1)\,k$ canales y agrega $k$ nuevos. El recorrido de formas de arriba sale exactamente así:

| Etapa | Capas | Canales que entran → salen | Mapa |
|---|---|---|---|
| Tallo (conv 7×7, max pool) | — | 3 → 64 | 56×56 |
| Bloque 1 + transición | 6 | 64 → 64 + 6·32 = 256 → 128 | 56×56 → 28×28 |
| Bloque 2 + transición | 12 | 128 → 512 → 256 | 28×28 → 14×14 |
| Bloque 3 + transición | 24 | 256 → 1024 → 512 | 14×14 → 7×7 |
| Bloque 4 | 16 | 512 → 1024 | 7×7 |
| Promedio global + lineal | — | 1024 → 14 | 1×1 |

Parámetros: $6{,}953{,}856$ de ImageNet $+\; 1024 \times 14 + 14 = 14{,}350$ de la capa nueva $= 6{,}968{,}206$.

**Por qué importa aquí:** la concatenación le da a cada capa un camino directo a todas las siguientes de su bloque, así que el gradiente llega sin desvanecerse (notebook 05, PASO 5), y la red reutiliza rasgos con pocos parámetros: 3 a 4 veces menos que los otros tres modelos con el mismo AUROC.

In [ ]:
# ===== PASO 9: evaluate: pérdida, AUROC y accuracy de un conjunto =====
# Qué hace: recorre un DataLoader con el modelo en modo evaluación y sin gradientes, junta
#   las probabilidades (sigmoide de los logits) y devuelve la pérdida, el AUROC promedio
#   de las 14 enfermedades, la accuracy con umbral 0.5 y las probabilidades.
# Por qué: es el eval_model de la clase 4 adaptado a 14 salidas. model.eval() hace que
#   BatchNorm use sus promedios guardados y torch.no_grad() ahorra memoria. El AUROC se
#   calcula sobre todo el conjunto, no por lote.
def auroc_por_clase(etiquetas, probs):
    """AUROC de cada enfermedad; NaN si en ese conjunto no hay positivos o no hay negativos."""
    return np.array([roc_auc_score(etiquetas[:, i], probs[:, i]) if 0 < etiquetas[:, i].sum() < len(etiquetas) else np.nan
                     for i in range(etiquetas.shape[1])])


def auroc_medio(etiquetas, probs):
    aucs = auroc_por_clase(etiquetas, probs)
    return float(np.nanmean(aucs)) if not np.all(np.isnan(aucs)) else 0.5


def evaluate(model, loader):
    """Devuelve (pérdida, AUROC promedio, accuracy con umbral 0.5, probabilidades) de un DataLoader."""
    model.eval()                                               # BatchNorm con sus promedios guardados
    probs, etiquetas, perdida = [], [], 0.0
    with torch.no_grad(), torch.autocast(device.type, dtype=torch.float16, enabled=USA_GPU):
        for images, labels in loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            logits = model(images).float()
            perdida += criterion(logits, labels).item() * labels.size(0)
            probs.append(torch.sigmoid(logits).cpu())          # probabilidad de cada enfermedad
            etiquetas.append(labels.cpu())
    probs, etiquetas = torch.cat(probs).numpy(), torch.cat(etiquetas).numpy()
    accuracy = float(((probs >= 0.5) == etiquetas).mean())     # aciertos sobre las 14 respuestas de cada radiografía
    return perdida / len(etiquetas), auroc_medio(etiquetas, probs), accuracy, probs

In [ ]:
# ===== PASO 10: train: el ciclo de entrenamiento escrito a mano =====
# Qué hace: en cada época: modo entrenamiento, y por cada lote forward en precisión mixta,
#   pérdida, zero_grad, backward, medición de la norma del gradiente y paso de AdamW; al
#   final de la época baja el learning rate (coseno), mide la muestra de entrenamiento y
#   la validación, guarda todo en history y en TensorBoard, guarda los pesos si el AUROC
#   de validación mejoró y para con early stopping si lleva 3 épocas sin mejorar.
# Por qué: es el train de la clase 8 (zero_grad -> backward -> step, con un registro por
#   paso) con la receta de 01a-01d. Es lo que Lightning hacía por dentro; aquí cada línea
#   está a la vista. Imprime una sola línea por época, sin barras: en Kaggle las barras
#   saturan la salida y detienen el notebook.
RUTA_MODELO = CARPETA_SALIDA / f"{MODELO}.ckpt"


def guardar_checkpoint(model, epoca, pasos):
    """Pesos de la mejor época con el formato de Lightning que lee el notebook 02 (state_dict con "model.*")."""
    torch.save({"state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
                "hyper_parameters": {"model_name": MODELO, "pos_weight": POS_WEIGHT, "optimizer_name": "AdamW",
                                     "optimizer_hparams": {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY}},
                "epoch": epoca - 1, "global_step": pasos}, RUTA_MODELO)


def train(model, name):
    """Entrena con la receta de 01a-01d y devuelve el diccionario history (por paso y por época)."""
    model.to(device)
    fusionado = {"fused": True} if USA_GPU and "fused" in inspect.signature(torch.optim.AdamW).parameters else {}
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY, **fusionado)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=T_MAX)
    scaler = torch.amp.GradScaler(enabled=USA_GPU)               # precisión mixta: escala la pérdida para fp16
    writer = SummaryWriter(log_dir=str(CARPETA_SALIDA / MODELO / "lightning_logs" / "version_0"))
    history = {k: [] for k in ["loss", "grad_norm", "train_loss", "train_loss_eval", "train_auc", "train_acc",
                               "val_loss", "val_auc", "val_acc", "lr", "seg_por_epoca"]}
    mejor_auc, mejor_epoca, pasos, pasos_mejor = -1.0, 0, 0, 0
    if USA_GPU:
        torch.cuda.reset_peak_memory_stats()

    for epoch in range(1, NUM_EPOCHS + 1):
        inicio = time.time()
        model.train()                                            # BatchNorm con estadísticas del lote
        suma = 0.0
        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            with torch.autocast(device.type, dtype=torch.float16, enabled=USA_GPU):
                loss = criterion(model(images).float(), labels)  # forward y pérdida

            optimizer.zero_grad(set_to_none=True)                # borra los gradientes del paso anterior
            scaler.scale(loss).backward()                        # backward: gradientes de la pérdida escalada
            scaler.unscale_(optimizer)                           # gradientes en su escala real
            norma = torch.nn.utils.clip_grad_norm_(model.parameters(), float("inf"))   # inf: solo mide la norma
            scaler.step(optimizer)                               # paso de AdamW (se salta si hubo inf/NaN)
            scaler.update()

            history["loss"].append(loss.item())
            history["grad_norm"].append(norma.item())
            suma += loss.item()
            pasos += 1
        history["lr"].append(optimizer.param_groups[0]["lr"])   # el lr con que se entrenó esta época
        scheduler.step()                                         # coseno: lr de la siguiente época

        train_loss_eval, train_auc, train_acc, _ = evaluate(model, train_eval_loader)   # muestra fija, sin aumento
        val_loss, val_auc, val_acc, _ = evaluate(model, val_loader)
        history["train_loss"].append(suma / len(train_loader))  # promedio durante la época, con aumento
        history["train_loss_eval"].append(train_loss_eval)
        history["train_auc"].append(train_auc)
        history["train_acc"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_auc"].append(val_auc)
        history["val_acc"].append(val_acc)
        history["seg_por_epoca"].append(time.time() - inicio)   # entrenamiento + evaluación, como en 01a
        for etiqueta, valor in [("train_loss", history["train_loss"][-1]), ("val_loss", val_loss), ("val_auc", val_auc),
                                ("train_auc", train_auc), ("val_acc", val_acc), ("lr-AdamW", history["lr"][-1])]:
            writer.add_scalar(etiqueta, valor, epoch)            # las curvas que lee el PASO 2 del notebook 02

        if val_auc > mejor_auc:                                  # ModelCheckpoint: guarda la mejor época
            mejor_auc, mejor_epoca, pasos_mejor = val_auc, epoch, pasos
            guardar_checkpoint(model, epoch, pasos)
        print(f"[{name}] época {epoch}/{NUM_EPOCHS}: {history['seg_por_epoca'][-1]:.0f} s | loss train "
              f"{history['train_loss'][-1]:.4f} val {val_loss:.4f} | AUROC train {train_auc:.4f} val {val_auc:.4f} | "
              f"acc val {val_acc:.4f} | lr {history['lr'][-1]:.2e}", flush=True)
        if epoch - mejor_epoca >= PATIENCE:                      # EarlyStopping: 3 épocas sin mejorar
            print(f"[{name}] early stopping: {PATIENCE} épocas sin mejorar el AUROC de validación", flush=True)
            break

    writer.close()
    history["vram_pico_gb"] = torch.cuda.max_memory_allocated() / 1024**3 if USA_GPU else 0.0
    history["mejor_epoca"], history["pasos_hasta_la_mejor"] = mejor_epoca, pasos_mejor
    history["pasos_por_epoca"] = len(train_loader)
    return history

### 📝 Lo que pasa en cada paso y en cada época
**AdamW.** Con el gradiente $g_t$ de cada parámetro, AdamW lleva dos promedios móviles:
$$m_t = \beta_1 m_{t-1} + (1-\beta_1)\,g_t, \qquad v_t = \beta_2 v_{t-1} + (1-\beta_2)\,g_t^2$$
Los corrige al inicio, $\hat m_t = m_t/(1-\beta_1^t)$ y $\hat v_t = v_t/(1-\beta_2^t)$, y actualiza:
$$\theta_t = \theta_{t-1} - \eta_t\left(\frac{\hat m_t}{\sqrt{\hat v_t} + \epsilon} + \lambda\,\theta_{t-1}\right)$$
con $\beta_1 = 0.9$, $\beta_2 = 0.999$, $\epsilon = 10^{-8}$, $\lambda = 0.05$ y $\eta_0 = 10^{-4}$. El término $\lambda\theta$ va aparte del gradiente (decaimiento de pesos *desacoplado*), y dividir entre $\sqrt{\hat v_t}$ hace que el tamaño del paso casi no dependa de la escala del gradiente.

**Coseno.** El learning rate de la época $e$ es:
$$\eta_e = \frac{\eta_0}{2}\left(1 + \cos\frac{\pi\,(e-1)}{T_{\max}}\right), \qquad T_{\max} = 10$$

| Época | 1 | 2 | 3 | 4 | 5 | 6 | 7 | 8 | 9 | 10 |
|---|---|---|---|---|---|---|---|---|---|---|
| $\eta_e$ (×10⁻⁴) | 1.00 | 0.98 | 0.90 | 0.79 | 0.65 | 0.50 | 0.35 | 0.21 | 0.10 | 0.02 |

**Precisión mixta.** El forward se hace en 16 bits. Para que los gradientes pequeños no se vuelvan cero en 16 bits, `GradScaler` multiplica la pérdida por un factor $s$ (empieza en $2^{16} = 65{,}536$) antes del backward, y `unscale_` divide los gradientes entre $s$ antes del paso.

**Norma del gradiente.** `clip_grad_norm_` con `max_norm = inf` no recorta nada: solo calcula
$$\lVert g \rVert_2 = \sqrt{\textstyle\sum_j g_j^2}$$
sobre todos los parámetros. Con esta receta, en el notebook 03 la mediana fue 1.67.

**Early stopping.** Sea $e^* = \arg\max_{k \le e} \text{AUROC}^{\text{val}}_k$. El entrenamiento para en la época $e$ si $e - e^* \ge 3$, y se conservan los pesos de $e^*$. Con todos los datos:
- DenseNet-121 tuvo su mejor época en la 3 y paró en la 6.
- CoAtNet-0 tuvo su mejor época en la 6 y paró en la 9.
- DeiT-S y Swin-T tuvieron su mejor época en la 7 y llegaron al máximo de 10.

### 📝 El ciclo a mano contra la versión con Lightning (`01a_DenseNet121`)

| Lightning (01a) | Este notebook |
|---|---|
| `CXRModule.forward` (normaliza y llama a la red) | `ModeloNormalizado.forward` |
| `training_step` (forward y pérdida de un lote) | el cuerpo de `for images, labels in train_loader:` dentro de `train` |
| Lo que Lightning llama por dentro: `zero_grad`, `backward`, `step` | `optimizer.zero_grad()`, `scaler.scale(loss).backward()`, `scaler.step(optimizer)` |
| `configure_optimizers` (AdamW y coseno) | `optimizer = AdamW(...)` y `scheduler = CosineAnnealingLR(...)`, con `scheduler.step()` al final de cada época |
| `validation_step` + `on_validation_epoch_end` (AUROC del conjunto) | `evaluate(model, val_loader)` |
| `Trainer(max_epochs=10, precision="16-mixed")` | `for epoch in range(1, NUM_EPOCHS + 1)` con `torch.autocast` y `GradScaler` |
| `ModelCheckpoint(monitor="val_auc", mode="max")` | `if val_auc > mejor_auc: guardar_checkpoint(...)` |
| `EarlyStopping(monitor="val_auc", patience=3)` | `if epoch - mejor_epoca >= PATIENCE: break` |
| `LearningRateMonitor` y `self.log(...)` | `history[...]` y `writer.add_scalar(...)` |
| El callback `CostoComputacional` | `time.time()` por época y `torch.cuda.max_memory_allocated()` |
| `trainer.test(model, test_loader)` | `evaluate(modelo_final, test_loader)` |

Lightning ahorra estas líneas, pero las ejecuta igual. Con la misma semilla, los dos caminos siguen la misma receta. No dan resultados idénticos bit a bit porque consumen los números aleatorios en otro orden.

In [ ]:
# ===== PASO 11: Entrenar (o cargar lo ya entrenado) =====
# Qué hace: si en la carpeta de salida ya están los pesos y la historia de este modelo,
#   los carga; si no, fija la semilla, entrena y guarda la historia en
#   historia_<Modelo>.json.
# Por qué: así, si la sesión se corta después de entrenar, al volver a correr el notebook
#   no se repite el entrenamiento, igual que en 01a.
ARCHIVO_HISTORIA = CARPETA_SALIDA / f"historia_{MODELO}.json"
if RUTA_MODELO.exists() and ARCHIVO_HISTORIA.exists():
    print(f"Ya existe {RUTA_MODELO.name}: se carga su historia y no se reentrena")
    history = json.loads(ARCHIVO_HISTORIA.read_text(encoding="utf-8"))
else:
    set_seed(SEED)                                           # mismo punto de partida en cada corrida
    inicio_total = time.time()
    history = train(modelo, MODELO)
    ARCHIVO_HISTORIA.write_text(json.dumps(history), encoding="utf-8")
    print(f"Entrenamiento: {(time.time() - inicio_total) / 60:.1f} min")

In [ ]:
# ===== PASO 12: Pérdida y norma del gradiente en cada paso =====
# Qué hace: dibuja la pérdida de cada lote y la norma del gradiente, suavizadas con el
#   promedio móvil de 50 pasos (la función smooth de la clase 8). Las líneas punteadas
#   marcan el fin de cada época.
# Por qué: la pérdida por lote es muy ruidosa; el promedio móvil deja ver la tendencia. La
#   norma del gradiente muestra si el entrenamiento fue estable (sin picos) y cómo baja al
#   acercarse a un mínimo.
def smooth(values, window=50):
    """Promedio móvil, como en la clase 8 (descarta inf y NaN de los pasos que saltó GradScaler)."""
    values = torch.tensor([v for v in values if np.isfinite(v)], dtype=torch.float32)
    if len(values) == 0:                                   # todos los pasos saltados: no hay nada que promediar
        return values
    window = min(window, len(values))
    return values.unfold(0, window, 1).mean(dim=1)


fig, (ax_loss, ax_grad) = plt.subplots(1, 2, figsize=(15, 4))
ax_loss.plot(smooth(history["loss"]))
ax_grad.plot(smooth(history["grad_norm"]), color="C1")
for ax in (ax_loss, ax_grad):
    for e in range(1, len(history["val_auc"])):
        ax.axvline(e * history["pasos_por_epoca"], color="gray", linestyle=":", linewidth=1)
ax_loss.set(title="Pérdida de entrenamiento por paso (promedio móvil de 50)", xlabel="paso", ylabel="BCE con pos_weight", yscale="log")
ax_grad.set(title="Norma del gradiente por paso (promedio móvil de 50)", xlabel="paso", ylabel="norma", yscale="log")
plt.tight_layout()
plt.show()

**Cómo leer la gráfica**
- La pérdida baja rápido en la primera época, porque la capa nueva aprende desde cero, y después baja despacio, porque el resto de la red solo se ajusta.
- Si la pérdida sube de golpe o la norma tiene picos, el paso fue demasiado grande. Con $\eta = 10^{-4}$ no pasa: eso fue justo lo que descartó a $10^{-3}$ en el notebook 03.
- Cada línea punteada es un cambio de época: el learning rate baja un escalón (coseno) y la pérdida suele dar un pequeño salto, porque el orden de los lotes vuelve a barajarse.

In [ ]:
# ===== PASO 13: Entrenamiento contra validación por época =====
# Qué hace: dibuja, por época, la pérdida y el AUROC de entrenamiento (muestra fija sin
#   aumento) contra los de validación, la accuracy de validación con umbral 0.5 junto a la
#   de decir siempre 'sano', y el learning rate. La línea punteada marca la mejor época.
# Por qué: es plot_history de LSTM_TS (clase 8): cuando el entrenamiento sigue mejorando y
#   la validación se estanca, el modelo memoriza. La accuracy junto a 'siempre sano'
#   muestra por qué la accuracy engaña en este problema.
epocas = range(1, len(history["val_auc"]) + 1)
mejor_epoca = int(np.argmax(history["val_auc"])) + 1
fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
axes[0, 0].plot(epocas, history["train_loss_eval"], marker="o", label="entrenamiento (muestra sin aumento)")
axes[0, 0].plot(epocas, history["val_loss"], marker="o", label="validación")
axes[0, 0].plot(epocas, history["train_loss"], marker=".", linestyle=":", color="C0", label="entrenamiento (durante la época)")
axes[0, 0].axhline(lineas_base.loc["Mejor probabilidad constante", "Pérdida val"], color="gray", linestyle="--",
                   label="mejor probabilidad constante")
axes[0, 0].set(title="Pérdida (BCE con pos_weight)", yscale="log")
axes[0, 1].plot(epocas, history["train_auc"], marker="o", label="entrenamiento (muestra sin aumento)")
axes[0, 1].plot(epocas, history["val_auc"], marker="o", label="validación")
axes[0, 1].axhline(0.5, color="gray", linestyle="--", label="adivinar (0.5)")
axes[0, 1].set(title="AUROC promedio de las 14 enfermedades")
axes[1, 0].plot(epocas, history["val_acc"], marker="o", color="C1", label="validación (umbral 0.5)")
axes[1, 0].axhline(ACC_SIEMPRE_SANO, color="black", linestyle="--", label="decir siempre 'sano'")
axes[1, 0].set(title="Accuracy de validación", xlabel="época")
axes[1, 1].plot(epocas, history["lr"], marker="o", color="C2")
axes[1, 1].set(title="Learning rate (coseno)", xlabel="época")
for ax in axes.flat:
    ax.axvline(mejor_epoca, color="gray", linestyle=":")
    ax.set_xticks(list(epocas))
    if ax.get_legend_handles_labels()[0]:
        ax.legend(fontsize=8)
plt.suptitle(f"{MODELO}: entrenamiento contra validación (línea punteada = mejor época, la que se guarda)")
plt.tight_layout()
plt.show()

**Cómo leer la gráfica**
- **Pérdida.** Compara las dos líneas continuas: se miden igual, al final de la época y sin aumento. Cuando la de entrenamiento sigue bajando y la de validación se estanca o sube, empieza el sobreajuste. La línea punteada (promedio durante la época, con aumento) puede quedar arriba de la de validación en la primera época sin que sea un error. La línea gris es la pérdida de una regla que no mira la imagen: el modelo debe quedar muy por debajo.
- **AUROC.** La distancia entre entrenamiento y validación es la brecha de generalización. La época que se guarda es la del máximo de validación, no la última.
- **Accuracy.** Si queda por debajo de decir siempre "sano", no es que el modelo sea malo: con `pos_weight` está dispuesto a marcar enfermedades, y cada falsa alarma baja la accuracy aunque detecte más enfermos. Por eso se elige con el AUROC.
- **Learning rate.** Baja como un coseno: las últimas épocas casi no mueven los pesos.

In [ ]:
# ===== PASO 14: Costo y convergencia =====
# Qué hace: resume el costo del entrenamiento con las mismas llaves que el callback de 01a
#   (segundos por época, tiempo total, memoria máxima de GPU, épocas, mejor época y época
#   al 99%) y lo guarda en costo_<Modelo>.json. Agrega los pasos y minutos hasta la mejor
#   época y el learning rate que tenía en ella.
# Por qué: los lineamientos piden comparar costo y facilidad de convergencia. Contar
#   épocas engaña cuando los modelos tardan distinto por época: los pasos y los minutos
#   hasta la mejor época se comparan directo.
curva = np.array(history["val_auc"])
mejor = int(curva.argmax()) + 1
costo = {"seg_por_epoca": [round(s, 1) for s in history["seg_por_epoca"]],
         "val_auc_por_epoca": [round(a, 4) for a in history["val_auc"]],
         "seg_epoca_promedio": round(float(np.mean(history["seg_por_epoca"])), 1),
         "tiempo_total_min": round(sum(history["seg_por_epoca"]) / 60, 1),
         "vram_pico_gb": round(history["vram_pico_gb"], 2),
         "epocas_entrenadas": len(curva),
         "mejor_epoca": mejor,
         "epoca_99": int(np.argmax(curva >= 0.99 * curva.max())) + 1}      # primera época con el 99% del mejor AUROC
(CARPETA_SALIDA / f"costo_{MODELO}.json").write_text(json.dumps(costo, indent=2))

pd.Series({"Segundos por época": costo["seg_epoca_promedio"], "Memoria GPU máxima (GB)": costo["vram_pico_gb"],
           "Épocas entrenadas": costo["epocas_entrenadas"], "Mejor época": mejor, "Época al 99%": costo["epoca_99"],
           "Pasos hasta la mejor": history["pasos_hasta_la_mejor"],
           "Minutos hasta la mejor": round(sum(history["seg_por_epoca"][:mejor]) / 60, 1),
           "lr en la mejor época": f"{history['lr'][mejor - 1]:.2e}",
           "Tiempo total (min)": costo["tiempo_total_min"]}, name=MODELO).to_frame()

### 📝 Convergencia: épocas, pasos y minutos
- **Época al 99%:** $e_{99} = \min\{\,e : \text{AUROC}^{\text{val}}_e \ge 0.99 \cdot \max_k \text{AUROC}^{\text{val}}_k\,\}$. Es la primera época que ya está casi en el máximo.
- **Pasos hasta la mejor época:** $e^* \cdot \lfloor N/B \rfloor$, con $\lfloor 77{,}988/32 \rfloor = 2{,}437$ pasos por época con todos los datos.
- **DenseNet121 con Lightning** (`01a_DenseNet121.ipynb`, Tesla T4, todos los pacientes): 435 s por época, mejor época 3 (al 99% en la 2), 7,311 pasos y 21.8 min hasta la mejor, con un learning rate de $0.90\cdot10^{-4}$; 6 épocas en total (43.5 min) y 2.16 GB de memoria. Esta versión debe dar cifras parecidas: la receta y la máquina son las mismas.

In [ ]:
# ===== PASO 15: El mejor modelo: predicciones de validación y test =====
# Qué hace: crea el modelo sin pesos de ImageNet, le carga los de la mejor época, predice
#   validación y test y guarda las probabilidades (preds_<Modelo>_val.csv y _test.csv, con
#   las columnas de 01a) y resultados_<Modelo>.json.
# Por qué: el test se mide una sola vez, con el modelo que eligió la validación. Los CSV
#   son la entrada del notebook 02 (umbrales, ensemble, Grad-CAM, subgrupos), sin volver a
#   correr la red.
modelo_final = crear_modelo(pretrained=False)
modelo_final.load_state_dict(torch.load(RUTA_MODELO, map_location="cpu", weights_only=True)["state_dict"])
modelo_final.to(device)
_, val_auc, _, probs_val = evaluate(modelo_final, val_loader)
_, test_auc, _, probs_test = evaluate(modelo_final, test_loader)
y_val, y_test = df_val[CLASES].values, df_test[CLASES].values


def guardar_predicciones(probs, df_conjunto, conjunto):
    """Mismas columnas que 01a: Image Index, Patient ID, las 14 etiquetas y p_<enfermedad>."""
    salida = pd.concat([df_conjunto[["Image Index", "Patient ID"] + CLASES].reset_index(drop=True),
                        pd.DataFrame(probs, columns=[f"p_{c}" for c in CLASES])], axis=1)
    salida.to_csv(CARPETA_SALIDA / f"preds_{MODELO}_{conjunto}.csv", index=False)


guardar_predicciones(probs_val, df_val, "val")
guardar_predicciones(probs_test, df_test, "test")
resultado = {"modelo": MODELO, "val_auc": round(val_auc, 4), "test_auc": round(test_auc, 4),
             "parametros": sum(p.numel() for p in modelo_final.model.parameters()), **costo}
(CARPETA_SALIDA / f"resultados_{MODELO}.json").write_text(json.dumps(resultado, indent=2))
print(f"AUROC validación {val_auc:.4f} | AUROC test {test_auc:.4f} | archivos en {CARPETA_SALIDA}")

In [ ]:
# ===== PASO 16: AUROC de test con intervalo de confianza y por enfermedad =====
# Qué hace: remuestrea pacientes del test con reemplazo (N_BOOTSTRAP veces, semilla 42),
#   recalcula el AUROC promedio en cada remuestreo y toma los percentiles 2.5 y 97.5.
#   Después dibuja el AUROC de test de cada enfermedad.
# Por qué: un solo número no dice cuánto podría cambiar con otros pacientes. Se
#   remuestrean pacientes, no radiografías, porque las radiografías de una misma persona
#   se parecen entre sí. Es el método del notebook 02.
pacientes_test = df_test["Patient ID"].values
indices_por_paciente = pd.Series(np.arange(len(pacientes_test))).groupby(pacientes_test).apply(np.array).to_dict()
lista_pacientes = np.array(list(indices_por_paciente))
rng = np.random.default_rng(SEED)
remuestreos = []
for _ in range(N_BOOTSTRAP):
    idx = np.concatenate([indices_por_paciente[u] for u in rng.choice(lista_pacientes, size=len(lista_pacientes), replace=True)])
    remuestreos.append(auroc_medio(y_test[idx], probs_test[idx]))
IC = np.percentile(remuestreos, [2.5, 97.5])
print(f"AUROC de test {test_auc:.4f} | IC 95% por paciente: {IC[0]:.3f} a {IC[1]:.3f} ({N_BOOTSTRAP} remuestreos)")

auroc_enfermedad = pd.Series(auroc_por_clase(y_test, probs_test), index=CLASES).sort_values()
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(auroc_enfermedad.index, auroc_enfermedad.values, color="C0")
ax.axvline(0.5, color="gray", linestyle="--", label="adivinar (0.5)")
ax.axvline(test_auc, color="C3", linestyle=":", label=f"promedio ({test_auc:.3f})")
ax.set(title=f"{MODELO}: AUROC de test por enfermedad", xlabel="AUROC", xlim=(0.4, 1.0))
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### 📝 El AUROC y su intervalo de confianza
El AUROC de la enfermedad $c$ es la probabilidad de que el modelo le dé más puntaje a un enfermo que a un sano, tomados al azar:
$$\text{AUROC}_c = P\big(s(x^+) > s(x^-)\big) = \frac{1}{n_c^+\,n_c^-}\sum_{i \in +}\sum_{j \in -}\mathbf{1}\big[s_i > s_j\big]$$
Vale 0.5 para cualquier regla que no mire la imagen y 1 para un orden perfecto. No depende de un umbral ni de cuántos sanos haya, y por eso es la métrica principal. El reportado es el promedio de las 14 enfermedades.

**Bootstrap por paciente.** En cada remuestreo $b = 1,\dots,200$ se eligen 2,797 pacientes del test con reemplazo y se calcula $\text{AUROC}^{(b)}$. El intervalo de 95% es $\big[\,q_{2.5\%},\ q_{97.5\%}\,\big]$ de esos 200 valores.

**DenseNet121 con Lightning, todos los pacientes:** AUROC de validación 0.832 y de test 0.807, con IC 95% de 0.800 a 0.812. La caída de validación a test se repite en los cuatro modelos: el test oficial es otra población (notebook 02, PASO 8).

In [ ]:
# ===== PASO 17: F1 con un umbral por enfermedad y accuracy contra 'siempre sano' =====
# Qué hace: para cada enfermedad elige en validación el umbral que maximiza el F1 y lo
#   aplica tal cual al test. Reporta el F1 macro, la accuracy por enfermedad y la exacta
#   (las 14 respuestas bien), con ese umbral, con 0.5 y para la regla de decir siempre
#   'sano'.
# Por qué: los lineamientos piden accuracy y F1. El umbral se elige en validación para no
#   inflar el test. Poner a 'siempre sano' en la misma tabla muestra que la accuracy no
#   separa a un buen modelo de uno que no detecta nada.
def umbrales_f1(y, p):
    """Para cada enfermedad, el umbral que maximiza el F1 en validación (0.5 si en validación no hay enfermos)."""
    umbrales = []
    for i in range(len(CLASES)):
        if y[:, i].sum() == 0:
            umbrales.append(0.5)
            continue
        precision, recall, cortes = precision_recall_curve(y[:, i], p[:, i])
        f1 = 2 * precision * recall / np.clip(precision + recall, 1e-9, None)
        umbrales.append(cortes[np.argmax(f1[:-1])])
    return np.array(umbrales)


def indicadores(y, pred):
    return {"F1 macro": f1_score(y, pred, average="macro", zero_division=0),
            "Accuracy por enfermedad": float((pred == y).mean()),
            "Accuracy exacta (14 de 14)": float((pred == y).all(axis=1).mean())}


UMBRALES = umbrales_f1(y_val, probs_val)
tabla_f1 = pd.DataFrame({f"{MODELO} (umbral elegido en validación)": indicadores(y_test, (probs_test >= UMBRALES).astype(int)),
                         f"{MODELO} (umbral 0.5)": indicadores(y_test, (probs_test >= 0.5).astype(int)),
                         "Siempre 'sano'": indicadores(y_test, np.zeros_like(y_test))}).T
tabla_f1.round(4)

### 📝 F1, accuracy y el umbral
Con el umbral $t_c$, la radiografía se marca con la enfermedad $c$ si $p_c \ge t_c$. Con los verdaderos positivos (VP), falsos positivos (FP) y falsos negativos (FN):
$$\text{precisión} = \frac{VP}{VP + FP}, \qquad \text{sensibilidad} = \frac{VP}{VP + FN}, \qquad F1 = \frac{2\cdot\text{precisión}\cdot\text{sensibilidad}}{\text{precisión} + \text{sensibilidad}}$$
El umbral de cada enfermedad es $t_c = \arg\max_t F1_c^{\text{val}}(t)$, y el F1 macro promedia las 14 enfermedades.
- La **accuracy por enfermedad** cuenta aciertos sobre las $14 \times N$ respuestas.
- La **exacta** pide acertar las 14 de una radiografía.

En el test, decir siempre "sano" da F1 = 0, pero 0.925 de accuracy por enfermedad y 0.387 de accuracy exacta. **DenseNet121 con Lightning:** F1 macro 0.326, accuracy por enfermedad 0.871 y exacta 0.197: mucho mejor en F1 y peor en accuracy que la regla que no detecta nada.

In [ ]:
# ===== PASO 18: RMSE de las probabilidades y escalado de Platt =====
# Qué hace: mide el RMSE entre probabilidad y etiqueta (la raíz del Brier score) del
#   modelo, de dos referencias (siempre la prevalencia de validación y siempre 0) y del
#   modelo calibrado con el escalado de Platt, ajustado en validación y aplicado al test.
# Por qué: el AUROC dice si el modelo ordena bien; el RMSE, si sus probabilidades son
#   creíbles. pos_weight infla las probabilidades; Platt las corrige sin reentrenar y sin
#   cambiar el AUROC, porque es una función creciente.
def rmse(p, y):
    return float(np.sqrt(np.mean((p - y) ** 2)))           # raíz del Brier score


def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6)
    return np.log(p / (1 - p))


def calibrar(p_val, y_val, p_test):
    """Escalado de Platt por enfermedad: sigmoide(a · logit(p) + b), ajustado en validación y aplicado al test."""
    salida = p_test.copy()
    for i in range(len(CLASES)):
        if 0 < y_val[:, i].sum() < len(y_val):
            lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p_val[:, [i]]), y_val[:, i])
            salida[:, i] = lr.predict_proba(logit(p_test[:, [i]]))[:, 1]
    return salida


probs_test_cal = calibrar(probs_val, y_val, probs_test)
tabla_rmse = pd.Series({"Referencia: siempre la prevalencia de validación": rmse(np.tile(y_val.mean(axis=0), (len(y_test), 1)), y_test),
                        "Referencia: siempre 0 ('sano')": rmse(np.zeros_like(y_test, dtype=float), y_test),
                        f"{MODELO} sin calibrar": rmse(probs_test, y_test),
                        f"{MODELO} con Platt": rmse(probs_test_cal, y_test)}, name="RMSE en test").round(4)
print(f"Probabilidad media predicha {probs_test.mean():.3f} | prevalencia media del test {y_test.mean():.3f} | "
      f"AUROC después de calibrar {auroc_medio(y_test, probs_test_cal):.4f} (igual que antes)")
tabla_rmse.to_frame()

### 📝 Brier, RMSE y escalado de Platt
$$\text{Brier} = \frac{1}{14N}\sum_{n=1}^{N}\sum_{c=1}^{14}\big(p_{n,c} - y_{n,c}\big)^2, \qquad \text{RMSE} = \sqrt{\text{Brier}}$$
Es el RMSE de `LSTM_TS` (clase 8) aplicado a probabilidades, y la referencia "siempre la prevalencia" hace el papel de la persistencia de esa clase. El escalado de Platt ajusta en validación dos números por enfermedad:
$$p' = \sigma\big(a_c \cdot \operatorname{logit}(p) + b_c\big), \qquad \operatorname{logit}(p) = \log\frac{p}{1-p}$$
Como $a_c > 0$, el orden de los pacientes no cambia y el AUROC queda igual.

En el test, la referencia de la prevalencia da 0.260 y decir siempre 0 da 0.275. **DenseNet121 con Lightning:** 0.293 sin calibrar, peor que las dos referencias porque `pos_weight` infla las probabilidades, y 0.240 con Platt, mejor que ambas.

In [ ]:
# ===== PASO 19: AUPRC y un punto de operación de 90% de sensibilidad =====
# Qué hace: calcula el AUPRC de cada enfermedad, la sensibilidad y especificidad con el
#   umbral del F1, y un punto de tamizaje: en validación, el umbral que deja arriba al 90%
#   de los enfermos de cada enfermedad, aplicado tal cual al test.
# Por qué: el AUPRC se mide contra la prevalencia y no contra 0.5, así que muestra lo
#   difícil que es encontrar las enfermedades raras. El punto de 90% responde la pregunta
#   clínica: cuántos sanos se marcan para no perder casi a ningún enfermo.
def sensibilidad_especificidad(y, pred):
    """Por enfermedad: verdaderos positivos / enfermos y verdaderos negativos / sanos."""
    with np.errstate(invalid="ignore", divide="ignore"):
        vp, fn = ((pred == 1) & (y == 1)).sum(0), ((pred == 0) & (y == 1)).sum(0)
        vn, fp = ((pred == 0) & (y == 0)).sum(0), ((pred == 1) & (y == 0)).sum(0)
        return vp / (vp + fn), vn / (vn + fp)


umbral_90 = np.array([np.quantile(probs_val[y_val[:, i] == 1, i], 0.10) if y_val[:, i].sum() else 0.5
                      for i in range(len(CLASES))])          # 90% de los enfermos de validación queda arriba
sens_90, esp_90 = sensibilidad_especificidad(y_test, (probs_test >= umbral_90).astype(int))
sens_f1, esp_f1 = sensibilidad_especificidad(y_test, (probs_test >= UMBRALES).astype(int))
auprc = np.array([average_precision_score(y_test[:, i], probs_test[:, i]) if y_test[:, i].sum() else np.nan
                  for i in range(len(CLASES))])
print(f"AUPRC macro {np.nanmean(auprc):.3f} (un clasificador al azar: {y_test.mean():.3f}) | umbral del F1: "
      f"sensibilidad {np.nanmean(sens_f1):.3f}, especificidad {np.nanmean(esp_f1):.3f} | umbral de 90% en validación: "
      f"sensibilidad {np.nanmean(sens_90):.3f}, especificidad {np.nanmean(esp_90):.3f}")
pd.DataFrame({"AUROC": auroc_por_clase(y_test, probs_test), "AUPRC": auprc, "Prevalencia test": y_test.mean(axis=0),
              "Sensibilidad (90% en val)": sens_90, "Especificidad (90% en val)": esp_90}, index=CLASES).round(3)

### 📝 AUPRC y punto de operación
El AUPRC (*average precision*) es el área bajo la curva de precisión contra sensibilidad:
$$\text{AUPRC} = \sum_n (R_n - R_{n-1})\,P_n$$
donde $P_n$ y $R_n$ son la precisión y la sensibilidad en el umbral $n$. Un clasificador al azar obtiene la prevalencia, 0.075 en promedio en el test. En Pneumonia (prevalencia 0.019), un AUROC de 0.73 corresponde a un AUPRC cercano a 0.05.

**Punto de operación de tamizaje.** El umbral de la enfermedad $c$ es el percentil 10 de los puntajes de los enfermos de validación:
$$t_c = q_{10\%}\big(\{\,p_{n,c} : y_{n,c} = 1\,\}\big)$$
Así, en validación, 90% de los enfermos queda arriba. Se aplica tal cual al test.

**DenseNet121 con Lightning:** AUPRC macro 0.278; con el umbral de 90%, sensibilidad 0.953 y especificidad 0.361 en test. Sirve para priorizar estudios, no para diagnosticar.

In [ ]:
# ===== PASO 20: Resumen del modelo =====
# Qué hace: junta en una tabla las métricas de este notebook (AUROC con su intervalo, F1,
#   accuracy, RMSE antes y después de calibrar, AUPRC, punto de operación y costo) y, si
#   existe el resultado de la versión con Lightning, lo pone al lado. La guarda en
#   resumen_<Modelo>.csv.
# Por qué: es la ficha del modelo que se lleva a la comparación entre los cuatro (notebook
#   02). Ponerla junto a la versión con Lightning muestra si el ciclo a mano reproduce la
#   misma receta.
fila_f1 = tabla_f1.iloc[0]
resumen = {"AUROC validación": round(val_auc, 4), "AUROC test": round(test_auc, 4),
           "IC 95% (por paciente)": f"{IC[0]:.3f} a {IC[1]:.3f}",
           "F1 macro (umbral de validación)": round(fila_f1["F1 macro"], 4),
           "Accuracy por enfermedad": round(fila_f1["Accuracy por enfermedad"], 4),
           "Accuracy exacta": round(fila_f1["Accuracy exacta (14 de 14)"], 4),
           "RMSE sin calibrar": tabla_rmse[f"{MODELO} sin calibrar"], "RMSE con Platt": tabla_rmse[f"{MODELO} con Platt"],
           "AUPRC macro": round(float(np.nanmean(auprc)), 4),
           "Sensibilidad / especificidad (90% en val)": f"{np.nanmean(sens_90):.3f} / {np.nanmean(esp_90):.3f}",
           "Parámetros": f"{resultado['parametros']:,}", "Segundos por época": costo["seg_epoca_promedio"],
           "Mejor época": costo["mejor_epoca"], "Pasos hasta la mejor": history["pasos_hasta_la_mejor"],
           "Memoria GPU (GB)": costo["vram_pico_gb"], "Tiempo total (min)": costo["tiempo_total_min"]}
tabla_resumen = pd.DataFrame({"Esta versión (ciclo a mano)": resumen})
archivo_lightning = Path("../resultados_kaggle/saved_models") / f"resultados_{MODELO}.json"
if archivo_lightning.exists():
    l = json.loads(archivo_lightning.read_text())
    tabla_resumen["Lightning (01a-01d, Tesla T4)"] = pd.Series({
        "AUROC validación": l["val_auc"], "AUROC test": l["test_auc"], "Parámetros": f"{l['parametros']:,}",
        "Segundos por época": l["seg_epoca_promedio"], "Mejor época": l["mejor_epoca"],
        "Memoria GPU (GB)": l["vram_pico_gb"], "Tiempo total (min)": l["tiempo_total_min"]})
tabla_resumen.to_csv(CARPETA_SALIDA / f"resumen_{MODELO}.csv")
tabla_resumen

**Takeaways**
- Lightning ahorra líneas, pero el ciclo es siempre el mismo: `zero_grad` → forward y pérdida → `backward` → `step`, con `model.train()` para entrenar y `model.eval()` con `torch.no_grad()` para medir.
- Siempre hay que comparar contra líneas base. Decir siempre "sano" gana en accuracy y pierde en AUROC y F1: la métrica correcta depende del problema.
- Todo lo que se decide (la época, los umbrales del F1, la calibración, el umbral de 90%) se decide en validación; el test solo se mide.
- Un número del test necesita su intervalo. Con una sola semilla, diferencias menores al ancho del intervalo no se pueden separar.

### 📝 Conclusión final
Este notebook entrena DenseNet-121 hecha a mano con la receta común del proyecto y lo evalúa con todas las métricas del artículo:
- el AUROC con su intervalo por paciente y por enfermedad;
- el F1 y la accuracy contra decir siempre "sano";
- el RMSE antes y después de calibrar;
- el AUPRC y un punto de operación clínico.

Además deja los archivos que el notebook 02 usa para comparar los cuatro modelos.

**Lo que se espera, según la versión con Lightning** (`01a_DenseNet121.ipynb`, todos los pacientes, Tesla T4): AUROC de test 0.807 (IC 95% 0.800 a 0.812), F1 macro 0.326, RMSE 0.293 sin calibrar y 0.240 con Platt, AUPRC 0.278, 6,968,206 parámetros, 435 s por época y mejor época 3. Si esta versión da cifras dentro del intervalo, el ciclo a mano reproduce la receta; las diferencias pequeñas vienen del orden en que se consumen los números aleatorios.

**Lo que muestra esta arquitectura:** la CNN más ligera del proyecto (7.0 M de parámetros) empata con modelos de 3 a 4 veces más parámetros. Con todos los datos llegó a su mejor época en la mitad de pasos que los demás, aunque esa ventaja desaparece con el 25% de los pacientes.